In [1]:
from lmetk.case_studies.cmapss import CMAPSSDataset
from lmetk.case_studies.ncmapss import NCMAPSSDataset
from pathlib import Path    
import hydra

%load_ext autoreload
%autoreload 2

In [2]:
ds = CMAPSSDataset("~/datasets/CMAPSS", 
                    concepts=["HPC", "Fan"])

sample = ds[100]
for id_t, t in enumerate(sample):
    print(f"Tensor {id_t} shape: {t.shape}")

Using linear RUL.
Used concepts: Index(['HPC', 'Fan'], dtype='object')
(20631, 24)


/Users/rtheiler/Workspaces/work/TabPHM/lmetk/case_studies/cmapss.py:32: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  self.df_X = pd.read_csv(os.path.join(path, f"{mode}_{n_DS}.txt"), sep=" ", header=None, index_col=False,


Tensor 0 shape: torch.Size([24])
Tensor 1 shape: torch.Size([])
Tensor 2 shape: torch.Size([2])


In [9]:
# from lmetk.data.datasets.cmapss import CMAPSSDataSource
from lmetk.data.datasets.abstract_classes import AbstractTimeSeriesDataSource


In [2]:

from box import Box
import hydra

import importlib
import lmetk.data.datasets.cmapss
from lmetk.data.datasets.rul_datamodule import ConceptRULDataset, SimlifiedRulDataModule
importlib.reload(lmetk.data.datasets.cmapss)


%load_ext autoreload
%autoreload 2

data_name = 'cmapss_refactored'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}', f"paths=rt_local"], 
        return_hydra_config=True, 
    )

    print(f"Config file: {parameters.dataset.loader}")

    data_source = hydra.utils.instantiate(parameters.dataset.loader)

type(data_source)
dd = data_source._load_data()
data_source._generate_splits()
data_source._apply_transformations()
data_source.df_X


in_box = Box({k: v.train for k, v in data_source.data_dict.items()})
ds_train = ConceptRULDataset(in_box, window_size=1, stride=1,)

in_box = Box({k: v.train for k, v in data_source.data_dict.items()})
ds_val = ConceptRULDataset(in_box, window_size=1, stride=1,)

in_box = Box({k: v.train for k, v in data_source.data_dict.items()})
ds_test = ConceptRULDataset(in_box, window_size=1, stride=1,)


sample = next(iter(ds_train))  
for id_t, t in enumerate(sample):
    print(f"Tensor {id_t} shape: {t.shape}")


datasets = Box(dict(
    train=ds_train,
    val=ds_val,
    test=ds_test,
))

dm = SimlifiedRulDataModule(datasets=datasets, batch_size=32)

sample = next(iter(dm.train_dataloader()))  
for id_t, t in enumerate(sample):
    print(f"Tensor {id_t} shape: {t.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Config file: {'_target_': 'lmetk.data.datasets.cmapss.CMAPSSDataSource', 'load_arguments': {'data_folder': 'CMAPSS', 'path': '${paths.data_dir}/${dataset.loader.load_arguments.data_folder}', 'n_DS': 'FD001', 'mode': 'train', 'units': None, 'concepts': ['HPC', 'Fan'], 'combined_concepts': False, 'binary_concepts': True, 'RUL': 'linear', 'include_healthy': True, 'window_size': 1, 'stride': 1}, 'data_splitter': {'_target_': 'lmetk.data.preprocessing.SimpleSplitter', 'train': 8640, 'val': 2880, 'test': None, 'seq_len': 384, 'label_len': 96, 'pred_len': 96}, 'scalers': {'features': {'_target_': 'lmetk.data.datasets.cmapss.CMAPSSScaler'}, 'labels': {'_target_': 'lmetk.data.datasets.cmapss.ConstantScaler', 'factor': 0.01}}}
Using linear RUL.
Used concepts: Index(['HPC', 'Fan'], dtype='object')
Tensor 0 shape: torch.Size([24])
Tensor 1 shape: torch.Size([])
Tensor 2 shape: torch.Size([2])
Tensor 0 shape: to

TypeError: RulDataModule.__init__() got an unexpected keyword argument 'datasets'

In [18]:
for k, v in data_source.data_dict.items():
    print(f"Key: {k}")
    print(f"Type: {type(v)}")
    print() 

Key: features
Type: <class 'box.box.Box'>

Key: timestamps
Type: <class 'box.box.Box'>

Key: health_states
Type: <class 'box.box.Box'>

Key: unit
Type: <class 'numpy.ndarray'>

Key: labels
Type: <class 'box.box.Box'>

Key: concepts
Type: <class 'box.box.Box'>



In [15]:
data_source.data_dict.unit.shape

(20631,)

In [17]:
data_source.data_dict.labels.train.shape

(8640,)

In [ ]:
def print_tensor_shapes(d, prefix=""):
    for k, v in d.items():
        if isinstance(v, dict):
            print(f"{prefix}{k}:")
            print_tensor_shapes(v, prefix + "  ")
        elif hasattr(v, "shape"):
            print(f"{prefix}{k}: shape {v.shape}")
        else:
            print(f"{prefix}{k}: type {type(v)}")

print_tensor_shapes(data_source.data_dict)

features:
  train: shape (8640, 27)
  val: shape (3264, 27)
  test: shape (9495, 27)
timestamps:
  train: shape (8640,)
  val: shape (3264,)
  test: shape (9495,)
health_states:
  train: shape (8640,)
  val: shape (3264,)
  test: shape (9495,)
labels:
  train: shape (8640,)
  val: shape (3264,)
  test: shape (9495,)
concepts:
  train: shape (8640, 2)
  val: shape (3264, 2)
  test: shape (9495, 2)


In [ ]:
data_source._load_data()
data_source.X
data_source.scaler

Using linear RUL.
Used concepts: Index(['HPC', 'Fan'], dtype='object')


In [ ]:
ds = NCMAPSSDataset(Path("~/datasets/N-CMAPSS").expanduser(), 
                        concepts=["HPC", "HPT"])
                        
sample = next(iter(ds))  
for id_t, t in enumerate(sample):
    print(f"Tensor {id_t} shape: {t.shape}")

Using linear RUL.
Used concepts: Index(['HPC', 'HPT'], dtype='object')
Tensor 0 shape: torch.Size([18, 50])
Tensor 1 shape: torch.Size([])
Tensor 2 shape: torch.Size([2])


/Users/rtheiler/Workspaces/work/load_management_evaluation/lmetk/case_studies/ncmapss.py:266: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return torch.Tensor(x), torch.Tensor(self.df_Y.iloc[i * self.stride]).squeeze(), torch.Tensor(self.concepts.iloc[i * self.stride].to_numpy())


# Test the RUL Dataset repo

In [ ]:
import torch
import rul_datasets

cmapss_fd1 = rul_datasets.CmapssReader(fd=1)


dm = rul_datasets.RulDataModule(cmapss_fd1, batch_size=32)
dm.prepare_data() 


dm.setup()



In [ ]:

sample = next(iter(dm.train_dataloader()))  
for id_t, t in enumerate(sample):
    print(f"Tensor {id_t} shape: {t.shape}")

Tensor 0 shape: torch.Size([32, 14, 30])
Tensor 1 shape: torch.Size([32])


/Users/rtheiler/Workspaces/work/TabPHM/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [3]:
class Test:
    _data = "1"


t = Test()
t._data = "2"


t2 = Test()
t2._data = "3"


t._data

'2'